In [1]:
import os
import sys
import json
from pathlib import Path
from ast import literal_eval

os.chdir("/home/kaariaa3/mscthesis/")
sys.path.append("./src/")  # Add module directory to path

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from utils.tools import aggregate_results

In [2]:
def get_suite(row):

    n_demos = row["number of demonstrations"]
    type_demos = row["type of demonstrations"][0:3]
    instr = "impl" if row["use instructions"] == "no" else "expl"

    return f"{n_demos}-{type_demos}-{instr}"

def capitalize(s):
    return s[0].upper() + s[1:]

def order_suites(strings):
    return sorted(strings, key=lambda s: (
        s.split('-')[2],  # implicit-excplict
        s.split('-')[0],  # num demos
        s.split('-')[1],  # demo type
    ))

def order_models(strings):
    return sorted(strings, key=lambda s: (
        int(s.split('-')[1][:-1]),  # Param count, B dropped
        #s.split('-')[0],  # Model name
    ), reverse=True)

def significance_label(x):
    return (
        ""
        if np.isnan(x)
        else "ns" if x < 1 
        else int(x) * "*"
    )

def format_p(x):
    return (
        "" 
        if np.isnan(x)
        else "<0.01"
        if x < 0.01
        else "%.3g" % x
    )

In [3]:
### Load F1 data

# F1 column names in dataset
metric_cols_f1 = ["theme f1 mean", "topic f1 mean", "concept f1 mean"]

# New column names
metric_cols = ["Theme F1", "Topic F1", "Concept F1", "Sum F1"]

# Labels
labels = ["Theme", "Topic", "Concept", "Sum"]

# Read csv
res = pd.read_csv("./data/metrics_v3.csv", sep=";")
res = res.sort_values(by=["use instructions", "number of demonstrations", "type of demonstrations"])

# Create suite column
res["suite"] = res.apply(get_suite, axis=1)

# Drop irrelevant columns + rename
f1s = res[["model", "suite"] + metric_cols_f1]

# Create sum column
f1s = f1s.assign(
    **{"Sum F1" : f1s[metric_cols_f1].sum(axis=1)}
)
f1s = f1s.rename(columns={metric_cols_f1[i]: metric_cols[i] for i in range(len(metric_cols_f1))})

# Capitalize column names
f1s.columns = [capitalize(col) for col in f1s.columns]

# Load McNemar scores 
path = "./data/mcnemar.json"
mcnemar = pd.read_json(path)

# Take only relevant columns
mcnemar.columns = [" ".join(literal_eval(tup)).strip() for tup in mcnemar.columns]
mcnemar = mcnemar[["Model", "Suite"] + [col for col in mcnemar.columns if any(s in col for s in ["Holm", "$S$"])]]

# Merge dfs on model and suite
f1s_with_p_values = f1s.merge(mcnemar, left_on=["Model", "Suite"], right_on=["Model", "Suite"])
f1s_with_p_values = f1s_with_p_values.set_index(["Model", "Suite"])
f1s_with_p_values = f1s_with_p_values.sort_index(axis=0)


# Unused, mcnemar data contains calculations for overall accuracy
#f1s_with_p_values = f1s_with_p_values.assign(
#    **{
#        "Sum Holm $p$": pd.Series([np.nan] * len(f1s_with_p_values)), # Cant be calculated
#        "Sum $S$": f1s_with_p_values[
#        [
#            f"{c.split(" ")[0]} $S$"
#            for c in metric_cols[:-1]
#        ]
#    ].sum(axis=1)
#    }
#)


f1s_with_p_values.columns = pd.MultiIndex.from_tuples([(col[ : col.index(" ")], col[col.index(" ") + 1 : ]) for col in f1s_with_p_values.columns])
f1s_with_p_values = f1s_with_p_values.sort_index(axis=1)

In [4]:
### Create table showing top 10 model, suite combinations for each label 
# IN USE!
top10s = [
    f1s_with_p_values
        .sort_values(by=(col, "F1"), ascending=False)[:10].loc[:, (col, slice(None))]
    for col in labels
]

for i, df in enumerate(top10s):

    label = labels[i]

    # Reorder cols
    df = df.iloc[:, [1, 2, 0]]

    # Map s, p values
    s_cols = [col for col in df.columns if "$S$" in col]
    p_cols = [col for col in df.columns if "Holm $p$" in col]

    df[s_cols] = df[s_cols].map(significance_label)
    df[p_cols] = df[p_cols].map(format_p)

    # Move index to columns
    names = df.index.names  
    df = df.reset_index()
    df.columns = pd.MultiIndex.from_arrays(
        [
            [label] * len(df.columns),
            list(names) + list(df.columns.get_level_values(1)[2:])
        ]
    )

    latex = df.to_latex(
        index=False,
        float_format="%.3g",
        column_format="ll|rrr",
        multicolumn_format="c",
        na_rep=""
    )
    
    print(latex)

\begin{tabular}{ll|rrr}
\toprule
\multicolumn{5}{c}{Theme} \\
Model & Suite & F1 & Holm $p$ & $S$ \\
\midrule
Llama-70B & 6-pos-expl & 0.943 & 1 & ns \\
Llama-70B & 0-non-expl & 0.939 &  &  \\
Llama-70B & 6-mix-expl & 0.939 & 1 & ns \\
Qwen3-32B & 6-mix-expl & 0.938 & 0.039 & * \\
Llama-70B & 6-neg-expl & 0.938 & 1 & ns \\
Qwen3-32B & 6-pos-expl & 0.936 & 0.15 & ns \\
Qwen3-32B & 1-pos-expl & 0.931 & 1 & ns \\
Llama-70B & 1-pos-expl & 0.928 & 0.0184 & * \\
Qwen3-32B & 6-neg-expl & 0.927 & 1 & ns \\
Qwen3-32B & 0-non-expl & 0.926 &  &  \\
\bottomrule
\end{tabular}

\begin{tabular}{ll|rrr}
\toprule
\multicolumn{5}{c}{Topic} \\
Model & Suite & F1 & Holm $p$ & $S$ \\
\midrule
Llama-70B & 0-non-expl & 0.977 &  &  \\
Llama-70B & 6-mix-expl & 0.973 & 1 & ns \\
Llama-70B & 6-neg-expl & 0.971 & 0.716 & ns \\
Llama-70B & 6-pos-expl & 0.971 & 0.73 & ns \\
Qwen3-32B & 6-mix-expl & 0.969 & 1 & ns \\
Qwen3-32B & 0-non-expl & 0.964 &  &  \\
Qwen3-32B & 6-neg-expl & 0.962 & 1 & ns \\
Llama-70B & 1-pos

In [6]:
### Create one table per label, where best of each model is presented
# In Use!

# Get F1 scores of baselines
baselines = f1s_with_p_values.groupby(by=["Model", "Suite"]).first().loc[(slice(None), "0-non-expl"), (slice(None), ["F1"])]
baselines.index = baselines.index.get_level_values(0)

tops_per_model = [
    f1s_with_p_values
        .loc[ # Take max F1 row for each model and label, take only columns corresponding to label 
            f1s_with_p_values.groupby('Model').idxmax()[(col, "F1")],
            (col, slice(None))
        ]
    for col in labels
]

for i, df in enumerate(tops_per_model):
   
    label = labels[i]

    # Sort models
    df = df.sort_values(by=(label, "F1"), ascending=False)

    # Calculate difference of best - baseline while maintaining indexing
    bl = baselines[label]["F1"]

    df[label, "Baseline"] = df.index.get_level_values("Model").map(bl)
    df[label, "$\\Delta$"] = df[label, "F1"] - df[label, "Baseline"]
        
    # Reorder cols
    df = df.iloc[:, [1, 3, 4, 2, 0]]

    # Rename (label, "F1") to (label, "Top F1")
    df = df.rename(columns={"F1": "Top F1"})
    
    # Map s, p values
    s_cols = [col for col in df.columns if "$S$" in col]
    p_cols = [col for col in df.columns if "Holm $p$" in col]

    df[s_cols] = df[s_cols].map(significance_label)
    df[p_cols] = df[p_cols].map(format_p)

    # Move index to columns
    names = df.index.names  
    df = df.reset_index()
    df.columns = pd.MultiIndex.from_arrays(
        [
            [label] * len(df.columns),
            list(names) + list(df.columns.get_level_values(1)[2:])
        ]
    )

    latex = df.to_latex(
        index=False,
        float_format="%.3g",
        column_format="ll|lllll",
        multicolumn_format="c",
        na_rep=""
    )
    
    print(latex)

\begin{tabular}{ll|llll}
\toprule
\multicolumn{7}{c}{Theme} \\
Model & Suite & Top F1 & Baseline & $\Delta$ & Holm $p$ & $S$ \\
\midrule
Llama-70B & 6-pos-expl & 0.943 & 0.939 & 0.00368 & 1 & ns \\
Qwen3-32B & 6-mix-expl & 0.938 & 0.926 & 0.0123 & 0.039 & * \\
Mistral-24B & 6-mix-expl & 0.889 & 0.852 & 0.0379 & 0.0166 & * \\
Llama-8B & 6-mix-expl & 0.868 & 0.697 & 0.171 & <0.01 & * \\
Qwen3-8B & 1-pos-expl & 0.794 & 0.785 & 0.00846 & 1 & ns \\
Mistral-7B & 6-mix-expl & 0.511 & 0.432 & 0.0788 & <0.01 & * \\
\bottomrule
\end{tabular}

\begin{tabular}{ll|llll}
\toprule
\multicolumn{7}{c}{Topic} \\
Model & Suite & Top F1 & Baseline & $\Delta$ & Holm $p$ & $S$ \\
\midrule
Llama-70B & 0-non-expl & 0.977 & 0.977 & 0 &  &  \\
Qwen3-32B & 6-mix-expl & 0.969 & 0.964 & 0.00464 & 1 & ns \\
Mistral-24B & 6-mix-expl & 0.913 & 0.888 & 0.025 & 0.201 & ns \\
Llama-8B & 6-pos-expl & 0.906 & 0.83 & 0.0762 & <0.01 & * \\
Qwen3-8B & 0-non-expl & 0.873 & 0.873 & 0 &  &  \\
Mistral-7B & 6-mix-expl & 0.706 & 